# 🚀 Hệ thống Quét CCCD/CMND ra Excel - Phiên bản Tốc Độ Bàn Thờ (Colab Pro-Tip)

Chào mừng bạn đến với phiên bản được **tối ưu hóa I/O và AI** chạy trên **Google Colab (GPU T4)**.
Hệ thống sẽ **giải nén ảnh trực tiếp cùng cấp với file zip** trên Google Drive của bạn, đồng thời xuất kết quả ra thư mục `Anh_CCCD_exports` ngay tại đó.

--- 
### 📌 QUY TRÌNH THỰC HIỆN:
1. Trên máy tính của bạn, hãy **nén tất cả ảnh CCCD thành 1 file .zip** (ví dụ: `Anh_CCCD.zip`).
2. Tải file `Anh_CCCD.zip` đó lên **Google Drive**.
3. Chạy lần lượt các bước từ 1 đến 5 bên dưới bằng cách bấm nút Play (Tam giác).


In [ ]:
# @title ⚙️ Kiểm tra cấu hình GPU T4 (Tùy chọn)
# Ô này giúp kiểm tra xem bạn đã bật GPU T4 thành công chưa. Nếu thấy báo lỗi, hãy vào menu Runtime > Change runtime type > Chọn T4 GPU.
!nvidia-smi

In [ ]:
# @title 📂 BƯỚC 1: Kết nối Google Drive
# Cấp quyền để Colab lấy file .zip ảnh từ Drive của bạn
from google.colab import drive
import os

print("Đang yêu cầu quyền truy cập Google Drive...")
drive.mount('/content/drive')
print("✅ Kết nối Google Drive thành công!")

In [ ]:
# @title 📦 BƯỚC 2: Tải Mã Nguồn & Tối ưu hóa Thư viện GPU
# @markdown Quá trình này sẽ cài đặt lõi xử lý ảnh tốc độ cao và `onnxruntime-gpu`. Mất khoảng 1-2 phút.

import os
from IPython.display import clear_output

print("1/4. Đang tải thư viện hệ thống...")
!apt-get update -qq
!apt-get install -y tesseract-ocr tesseract-ocr-vie libzbar0 libgl1-mesa-glx > /dev/null

print("2/4. Đang tải mã nguồn...")
%cd /content
if not os.path.exists('cccd-qr-excel'):
    !git clone https://github.com/vonguyendang/cccd-qr-excel.git
else:
    %cd cccd-qr-excel
    !git pull
    %cd ..

print("3/4. Đang cài đặt công cụ ép xung GPU (onnxruntime-gpu) & AI...")
%cd /content/cccd-qr-excel
!sed -i '/numpy<2.0.0/d' wizard/requirements.txt
!sed -i '/onnxruntime/d' deepdoc_vietocr/requirements.txt
!sed -i '/opencv-python/d' deepdoc_vietocr/requirements.txt

!pip install -r deepdoc_vietocr/requirements.txt pytesseract pyzbar opencv-python==4.9.0.80 opencv-python-headless==4.9.0.80 opencv-contrib-python==4.9.0.80 opencv-contrib-python-headless==4.9.0.80 openpyxl pillow-heif zxing-cpp==3.0.0 torch torchvision vietocr pycryptodomex setuptools pdfplumber onnxruntime-gpu==1.19.2 "numpy<2.0.0" > /dev/null
clear_output()
print("✅ Môi trường GPU T4 đã sẵn sàng ở mức năng lượng tối đa!")

In [ ]:
# @title 📥 BƯỚC 3: Giải nén file ảnh
# @markdown Nhập đường dẫn file **.ZIP** chứa ảnh CCCD trên Google Drive của bạn.
# @markdown Ví dụ: `/content/drive/MyDrive/CCCD_Test.zip`
file_zip_tren_drive = "/content/drive/MyDrive/CCCD_Test.zip" # @param {type:"string"}

import os

if not os.path.exists(file_zip_tren_drive):
    print(f"❌ Lỗi: Không tìm thấy file '{file_zip_tren_drive}'!")
    print("Hãy bấm vào tab Files bên trái, tìm file zip và copy lại đường dẫn.")
else:
    thu_muc_cha = os.path.dirname(file_zip_tren_drive)
    thu_muc_anh = os.path.join(thu_muc_cha, 'Anh_CCCD')
    print(f"1. Đang tạo thư mục giải nén cùng cấp với file zip: {thu_muc_anh}...")
    !rm -rf "{thu_muc_anh}"
    !mkdir -p "{thu_muc_anh}"
    print("2. Đang giải nén ảnh (có thể mất chút thời gian do giải nén trực tiếp trên Drive)...")
    !unzip -q -j "{file_zip_tren_drive}" -d "{thu_muc_anh}"
    print("✅ Giải nén hoàn tất! Đã sẵn sàng đưa vào GPU quét.")


In [ ]:
# @title 🚀 BƯỚC 4: Kích hoạt Quét AI
# @markdown Tiến trình sẽ đọc các ảnh trong thư mục vừa giải nén.

import os
thu_muc_cha = os.path.dirname(file_zip_tren_drive)
thu_muc_anh = os.path.join(thu_muc_cha, 'Anh_CCCD')
if not os.path.exists(thu_muc_anh) or len(os.listdir(thu_muc_anh)) == 0:
    print("❌ Lỗi: Thư mục chứa ảnh bị trống hoặc không tồn tại. Hãy chạy lại Bước 3.")
else:
    print("Đang nạp AI Models vào VRAM...")
    %cd /content/cccd-qr-excel/wizard
    
    # Cho phép người dùng tự nhập các tùy chọn quét
    !python3 main.py "{thu_muc_anh}"
    
    print("\n=========================================")
    print("✅ QUÉT HOÀN TẤT!")
    print("Chuyển qua Bước 5 để đóng gói kết quả.")
    print("=========================================")


In [ ]:
# @title 📦 BƯỚC 5: Đóng gói và Lưu kết quả trên Drive
# @markdown Kết quả xuất ra sẽ được nén lại thành 1 file ZIP (bao gồm Excel và các ảnh đã đổi tên) và tải về.

import shutil
import os
from google.colab import files

thu_muc_cha = os.path.dirname(file_zip_tren_drive)
export_dir = os.path.join(thu_muc_cha, 'Anh_CCCD_exports')
thu_muc_luu_tren_drive = os.path.join(thu_muc_cha, 'KetQua_CCCD')

if os.path.exists(export_dir) and len(os.listdir(export_dir)) > 0:
    if not os.path.exists(thu_muc_luu_tren_drive):
        os.makedirs(thu_muc_luu_tren_drive)
        
    # Nén toàn bộ thư mục Exports thành file ZIP
    zip_name = "Export_KetQua_CCCD"
    zip_path = f"/content/{zip_name}"
    print("Đang đóng gói file Excel, Log và các file Zip thành 1 file duy nhất...")
    shutil.make_archive(zip_path, 'zip', export_dir)
    
    dest_path = os.path.join(thu_muc_luu_tren_drive, f"{zip_name}.zip")
    print(f"Đang lưu file vào thư mục trên Drive: {dest_path}")
    shutil.copy2(f"{zip_path}.zip", dest_path)
    print("\n✅ Đã lưu thành công file ZIP lên Google Drive!")
    
    print("\n⬇️ Đang ra lệnh cho trình duyệt TỰ ĐỘNG TẢI XUỐNG file ZIP...")
    try:
        files.download(f"{zip_path}.zip")
        print("\n⚠️ LƯU Ý: Nếu trình duyệt của bạn chặn pop-up, quá trình tự động tải xuống có thể thất bại.")
        print(f"👉 Khi đó, bạn hãy tự mở thư mục '{thu_muc_luu_tren_drive}' trên Google Drive để lấy file nhé!")
    except Exception as e:
        print("\n❌ Không thể tự động tải file xuống.")
        print(f"👉 Bạn hãy tự mở thư mục '{thu_muc_luu_tren_drive}' trên Google Drive để lấy file nhé!")
else:
    print("❌ Không tìm thấy dữ liệu xuất ra. Có thể bạn chưa chạy Bước 4 thành công.")
